# vehicle0: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('30_vehicle0_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.788,93.242,92.662,93.987,97.683,95.835,98.427,99.539,94.515
smote,97.245,93.646,89.764,97.994,96.525,97.259,98.565,99.572,95.446
bsmote,97.319,92.755,87.166,99.244,95.444,97.344,98.642,99.597,95.037
ros,97.588,93.714,89.035,98.994,96.213,97.604,98.712,99.627,95.651
adasyn,97.671,93.747,88.956,99.250,96.138,97.694,98.635,99.605,95.709
casb,97.309,93.143,88.239,98.750,95.907,97.328,98.493,99.560,95.226
casb_relaxed,97.427,93.479,88.827,98.750,96.138,97.444,98.491,99.559,95.453
casb_edited,96.721,91.355,84.927,99.000,94.513,96.757,97.468,99.318,94.038
casb_ros,97.191,92.821,87.659,98.744,95.676,97.210,98.542,99.572,95.006


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,91.703,88.140,89.835,86.974,96.908,91.941,94.922,98.462,89.921
smote,92.681,85.307,78.816,93.237,92.197,92.717,93.443,98.163,88.994
bsmote,93.500,85.850,78.192,95.237,91.811,93.524,92.801,98.163,89.675
ros,92.942,86.107,80.288,92.987,92.967,92.977,93.622,98.288,89.524
adasyn,92.927,84.999,77.325,94.487,91.420,92.954,93.359,98.274,88.963
casb,92.844,84.462,76.147,94.987,90.803,92.895,92.957,98.132,88.653
casb_relaxed,92.420,84.142,76.348,93.987,90.957,92.472,92.269,98.075,88.281
casb_edited,91.475,81.842,72.501,94.237,88.868,91.552,88.650,97.380,86.659
casb_ros,92.301,83.978,76.178,93.737,90.959,92.348,90.889,97.755,88.139


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.737,93.636,93.972,93.481,98.146,95.813,98.544,99.545,94.687
smote,96.931,93.901,91.262,96.744,97.142,96.943,98.368,99.523,95.416
bsmote,96.954,93.512,90.145,97.250,96.679,96.964,98.517,99.522,95.233
ros,96.498,93.611,91.684,95.744,97.296,96.520,98.456,99.535,95.055
adasyn,96.972,94.027,91.528,96.750,97.220,96.985,98.480,99.523,95.500
casb,96.961,94.019,91.566,96.750,97.220,96.985,98.317,99.500,95.490
casb_relaxed,96.250,93.132,91.010,95.500,97.066,96.283,98.083,99.419,94.691
casb_edited,96.153,90.607,84.293,97.994,94.360,96.177,96.617,99.067,93.380
casb_ros,97.397,94.304,91.155,97.750,97.065,97.407,98.629,99.585,95.850


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,95.179,92.545,92.444,92.962,97.524,95.243,96.057,99.271,93.862
smote,96.283,92.558,89.296,96.224,96.368,96.296,96.000,99.267,94.421
bsmote,96.718,92.680,88.473,97.474,95.982,96.728,96.116,99.237,94.699
ros,96.823,93.217,89.686,97.224,96.444,96.834,96.264,99.280,95.020
adasyn,96.583,92.780,89.107,96.974,96.211,96.593,96.218,99.263,94.682
casb,96.614,93.083,89.866,96.724,96.522,96.623,96.198,99.270,94.849
casb_relaxed,96.487,92.936,89.806,96.474,96.522,96.498,96.284,99.286,94.712
casb_edited,96.870,92.001,86.257,98.731,95.056,96.893,93.890,98.845,94.436
casb_ros,96.526,93.062,90.065,96.474,96.599,96.537,96.181,99.266,94.794


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.602,91.891,92.228,91.851,97.565,94.708,96.988,99.204,93.246
smote,95.785,91.353,87.284,96.050,95.558,95.804,96.594,99.131,93.569
bsmote,96.123,91.200,85.994,97.301,94.979,96.140,96.519,99.130,93.661
ros,95.962,91.662,87.673,96.237,95.730,95.984,96.764,99.183,93.812
adasyn,96.038,91.388,86.729,96.865,95.247,96.056,96.673,99.166,93.713
casb,95.932,91.177,86.455,96.803,95.113,95.958,96.491,99.116,93.554
casb_relaxed,95.646,90.923,86.498,96.178,95.171,95.674,96.282,99.085,93.284
casb_edited,95.305,88.951,81.995,97.490,93.199,95.345,94.156,98.652,92.128
casb_ros,95.854,91.041,86.264,96.676,95.075,95.875,96.060,99.045,93.447


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,97.427,adasyn,97.671,-0.244,LOSS
1,f1,casb_relaxed,93.479,adasyn,93.747,-0.268,LOSS
2,precision,casb_relaxed,88.827,smote,89.764,-0.937,LOSS
3,recall,casb_edited,99.000,adasyn,99.250,-0.250,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,92.844,bsmote,93.500,-0.656,LOSS
1,f1,casb,84.462,ros,86.107,-1.645,LOSS
2,precision,casb_relaxed,76.348,ros,80.288,-3.940,LOSS
3,recall,casb,94.987,bsmote,95.237,-0.250,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,97.397,adasyn,96.972,0.425,WIN
1,f1,casb_ros,94.304,adasyn,94.027,0.276,WIN
2,precision,casb,91.566,ros,91.684,-0.118,LOSS
3,recall,casb_edited,97.994,bsmote,97.250,0.744,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,95.932,bsmote,96.123,-0.191,LOSS
1,f1,casb,91.177,ros,91.662,-0.485,LOSS
2,precision,casb_relaxed,86.498,ros,87.673,-1.175,LOSS
3,recall,casb_edited,97.490,bsmote,97.301,0.189,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.946857,0.913580,0.902439,0.925000,0.969231,0.947115,0.982902,0.994615,0.930219,None
1,42,1,svm,smote,0.75,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.986052,0.995769,0.953442,None
2,42,1,svm,bsmote,0.75,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.983622,0.995000,0.953442,None
3,42,1,svm,ros,1.00,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.981176,0.994615,0.953442,None
4,42,1,svm,adasyn,0.75,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.987601,0.996346,0.953442,None
5,42,1,svm,casb,0.50,0.980581,0.941176,0.888889,1.000000,0.961538,0.980769,0.980520,0.994231,0.960879,None
6,42,1,svm,casb_relaxed,0.75,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.979496,0.993846,0.953442,None
7,42,1,svm,casb_edited,0.75,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.971004,0.991923,0.953442,None
8,42,1,svm,casb_ros,0.75,0.976650,0.930233,0.869565,1.000000,0.953846,0.976923,0.977038,0.993462,0.953442,None
9,42,1,knn,original,0.50,0.887520,0.864865,0.941176,0.800000,0.984615,0.892308,0.949439,0.983077,0.876193,None
